In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split

DATA = Path("../data")
COLS = ["label", "id", "date", "query", "user", "tweet"]
df = pd.read_csv(DATA / "training.1600000.processed.noemoticon.csv",encoding="latin-1", header=None, names=COLS)
df = df[["label",   "id", "tweet"]].copy()
df["label"] = df["label"].map({0: 0, 4: 1})
df["lowercase_tweet"] = df["tweet"].str.strip().str.lower()
print(df["label"].value_counts())
df.head()

label
0    800000
1    800000
Name: count, dtype: int64


,label,id,tweet,lowercase_tweet
0,0,1467810369,"@switchfoot http://twitpic.com/2y1zl - Awww, t...","@switchfoot http://twitpic.com/2y1zl - awww, t..."
1,0,1467810672,is upset that he can't update his Facebook by ...,is upset that he can't update his facebook by ...
2,0,1467810917,@Kenichan I dived many times for the ball. Man...,@kenichan i dived many times for the ball. man...
3,0,1467811184,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire
4,0,1467811193,"@nationwideclass no, it's not behaving at all....","@nationwideclass no, it's not behaving at all...."


In [2]:
labels_per_tweet = df.groupby("lowercase_tweet")["label"].nunique()
conflicting = labels_per_tweet[labels_per_tweet > 1].index
print("tweets with conflicting labels:", len(conflicting))
df_clean = df[~df["lowercase_tweet"].isin(conflicting)]
df_nodup = df_clean.drop_duplicates(subset="lowercase_tweet", keep="first")
print(len(df), "→", len(df_clean), "→", len(df_nodup))

tweets with conflicting labels: 2378
1600000 → 1591325 → 1574322


In [3]:
df_sample = df_nodup.groupby("label").sample(n=120000, random_state=42)
print(len(df_sample))
print(df_sample["label"].value_counts())

240000
label
0    120000
1    120000
Name: count, dtype: int64


In [4]:
train_and_validation, test = train_test_split(
    df_sample,
    test_size=20000,
    stratify=df_sample["label"],
    random_state=42
)

train, validation = train_test_split(
    train_and_validation,
    test_size=20000,
    stratify=train_and_validation["label"],
    random_state=42
)

print(len(train), len(validation), len(test))
print(train["label"].value_counts())
print(validation["label"].value_counts())
print(test["label"].value_counts())


200000 20000 20000
label
0    100000
1    100000
Name: count, dtype: int64
label
1    10000
0    10000
Name: count, dtype: int64
label
0    10000
1    10000
Name: count, dtype: int64


In [5]:
SPLITS = DATA / "splits"
SPLITS.mkdir(parents=True, exist_ok=True)

COLS_TO_SAVE = ["label", "id", "tweet"]

train[COLS_TO_SAVE].to_csv(SPLITS / "train.csv", index=False)
validation[COLS_TO_SAVE].to_csv(SPLITS / "validation.csv", index=False)
test[COLS_TO_SAVE].to_csv(SPLITS / "test.csv", index=False)

check = pd.read_csv(SPLITS / "train.csv")
print("train vs val: ", train["lowercase_tweet"].isin(validation["lowercase_tweet"]).sum())
print("train vs test:", train["lowercase_tweet"].isin(test["lowercase_tweet"]).sum())
print("val vs test:  ", validation["lowercase_tweet"].isin(test["lowercase_tweet"]).sum())
print(check.shape)
check.head()

train vs val:  0
train vs test: 0
val vs test:   0
(200000, 3)


,label,id,tweet
0,0,2323739350,Never ever!! Eat an ice cream sandwich out of ...
1,0,2234329532,SA .. yet again
2,1,1834435622,@aussiemcflyfan aww really? maybe in a few yea...
3,0,2060875771,Im so tired!!! Baby sitting.
4,0,2189485033,"On my way to see about getting a bus, aren't c..."


## Summary: train / validation / test splits

### Cleaning (before splitting)
Starting point: 1,600,000 tweets (labels converted to 0 = negative, 1 = positive).

To find duplicates and conflicts, I compared a normalized copy of each tweet (lowercased, surrounding spaces removed), so case variants such as "good morning" and "Good morning" count as the same tweet. The saved files keep the original text.

1. **Conflicting labels:** 2,378 tweet texts appeared with both labels (e.g. "good morning"). No model can get both versions right, so I removed every copy of these texts: 8,675 rows in total.
2. **Duplicates:** I then removed duplicate texts, keeping the first copy of each: 17,003 rows removed. This prevents the same tweet from appearing in both training and test data (data leakage).

Cleaned dataset: 1,574,322 unique tweets.

### Sampling and splitting
- Balanced sample: 240,000 tweets, 120,000 per label (sampled per label with `random_state=42`).
- Split with `train_test_split`, stratified on the label, `random_state=42`:
  1. The test set was split off first, so it is fixed and independent of later choices.
  2. The remaining 220,000 tweets were split into training and validation.

| Set | Tweets | Negative | Positive |
|---|---|---|---|
| Training | 200,000 | 100,000 | 100,000 |
| Validation | 20,000 | 10,000 | 10,000 |
| Test | 20,000 | 10,000 | 10,000 |

Overlap check (on the normalized text): no tweet appears in more than one set. All pairwise overlaps are 0.

### Files
Saved in `data/splits/` (not committed to GitHub; recreate them by running this notebook):
- `train.csv`
- `validation.csv`
- `test.csv`

Each file has three columns: `label`, `id`, `tweet`.

### Roles
- **Training:** the models learn from these tweets.
- **Validation:** used to compare settings and choose the best model version.
- **Test:** used only once per model, at the end, for the final score.